# CCDP — Car Crash Fix-Amount Predictor (capstone submission)

> **Photo → identify car → detect damage → estimate cost (USD/INR).**

This is the single notebook that drives the project. It walks through
the four variants we built (A → B → C → D), the training pipeline for each
model, and ends with a live inference demo using the released v0.2.0 weights.

### How to read this notebook
- **Section 1**: setup, repo install, environment.
- **Section 2** *(identifier training)*: Stanford-Cars baseline → continue-train on VMMRdb.
- **Section 3** *(damage segmentation)*: YOLOv8-seg on CarDD (nc=6).
- **Section 3b** *(Path A extension)*: CarDD + HITL — optional, remove if not run.
- **Section 4** *(parts segmentation)*: 15-class car-parts YOLOv8-seg.
- **Sections 5–8**: Variant A → D, the methodological arc. Each shows what was
  added and what the previous variant was missing.
- **Section 9**: multi-car extension.
- **Section 10**: live inference — upload an image, get a costed report.
- **Section 11**: reproducibility checklist + metrics summary table.

### Conventions
- **All training cells are guarded by `RUN_TRAINING = False`.** Smoke values
  (epochs=1, batch=2, imgsz=320) are wired in by default. Production values
  used for the released v0.2.0 weights are documented one line away in
  comments — substitute and flip the flag to actually train.
- **All released weights** are fetched from the GitHub release v0.2.0 — no need
  to retrain anything to run the demo.
- **Datasets** are CC0 / permissive: CarDD, Stanford-Cars, VMMRdb (Kaggle CC0
  mirror), HITL (CC0), carparts (CC0).


## 1.1 Install + repo clone

In [ ]:
# If you're running this in Colab or Kaggle, clone the repo first.
# Locally, skip this cell and `cd` into the project root before launching jupyter.
import os, sys, pathlib

REPO = 'car-crash-fix-amount-predictor'
ON_COLAB  = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle')

if ON_COLAB or ON_KAGGLE:
    WORK = '/content' if ON_COLAB else '/kaggle/working'
    os.chdir(WORK)
    if not pathlib.Path(REPO).exists():
        os.system(f'git clone --depth 1 https://github.com/theDocWho/{REPO}.git')
    os.chdir(f'{WORK}/{REPO}')
    os.system('pip -q install -e .[ml]')

import ccdp
print(f'ccdp {ccdp.__version__ if hasattr(ccdp, "__version__") else ""} ok at', os.getcwd())

## 1.2 Environment + device check
We auto-pick CUDA → MPS → CPU. Training cells respect this. The demo at the
end runs comfortably on CPU.

In [ ]:
import torch
device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('torch:', torch.__version__, 'device:', device,
      torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 1.3 Fetch released production weights
All four production models are attached to the v0.2.0 GitHub release.
The demo and inference cells read from `checkpoints/production/`. Re-running
this is idempotent — already-downloaded files are skipped.

In [ ]:
import os, urllib.request, pathlib
REL = 'https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0'
PROD = pathlib.Path('checkpoints/production'); PROD.mkdir(parents=True, exist_ok=True)
ASSETS = {
    'identifier.pt':   f'{REL}/identifier.pt',
    'damage_cls.pt':   f'{REL}/damage_cls.pt',    # Variant A
    'damage_det.pt':   f'{REL}/damage_det.pt',    # Variant B
    'damage_seg.pt':   f'{REL}/damage_seg.pt',    # Variants C, D
    'parts_seg.pt':    f'{REL}/parts_seg.pt',     # Variant D
}
for name, url in ASSETS.items():
    dst = PROD / name
    if dst.exists():
        print(f'  {name:18s} ({dst.stat().st_size/1e6:.1f} MB) — present')
        continue
    print(f'  {name:18s} downloading…')
    try:
        urllib.request.urlretrieve(url, dst)
        print(f'    -> {dst.stat().st_size/1e6:.1f} MB')
    except Exception as e:
        print(f'    SKIPPED ({e}). Demo cells will warn.')

# Initialise the parts-cost catalog (3-tier pricing keyed by part × severity)
os.system('ccdp costing init || true')
print('done')

## 2. Identifier training (Stanford → VMMRdb)

**Goal:** identify the make/model/year of the photographed car so we can pick
the right cost segment (a Maruti bumper ≠ a BMW bumper).

### Pipeline
1. Train a ResNet-50 head from ImageNet → Stanford-Cars (196 classes). This is
   the warm-start baseline.
2. Swap the final `Linear(512→N)` head to the new label space, continue-train
   on VMMRdb (~9,170 make/model/year combinations), two-stage:
   - **Stage 1** (epochs 1–2): backbone frozen, lr 5e-4 — gets the new head
     off random init without disturbing pretrained features.
   - **Stage 2** (epochs 3–12): unfreeze layer3/4, lr 5e-5 — gentle fine-tune.

### `🚧 TBD — waiting on the VMMRdb Kaggle run`
The VMMRdb continue-train is currently running on Kaggle / Colab via
`notebooks/kaggle_train_identifier.ipynb` or `notebooks/colab_train_identifier.ipynb`.

When it finishes, this section will be filled in with:
- Final val acc on VMMRdb
- Stanford make-level anchor accuracy (forgetting proxy)
- Confusion matrix for the top-50 confused class pairs
- Released `identifier.pt` checksum

For now, the cell below shows the training command that produced the released
weights. It's guarded by `RUN_TRAINING=False`.

In [ ]:
# === Section 2: Identifier two-stage continue-train ===
RUN_TRAINING = False

# Production values used for the released v0.2.0 weights:
#   --dataset vmmrdb  --top-n 0  --batch-size 64  --num-workers 4
#   --epochs-stage1 2  --epochs-stage2 10
EPOCHS_STAGE1 = 1   # production: 2
EPOCHS_STAGE2 = 1   # production: 10
BATCH_SIZE    = 2   # production: 64
TOP_N         = 50  # production: 0  (= all ~9170 classes)

if RUN_TRAINING:
    cmd = (
        f'ccdp train identifier-continue --dataset vmmrdb '
        f'--top-n {TOP_N} --batch-size {BATCH_SIZE} --num-workers 2 '
        f'--epochs-stage1 {EPOCHS_STAGE1} --epochs-stage2 {EPOCHS_STAGE2}'
    )
    print('$', cmd); os.system(cmd)
else:
    print('Skipped — flip RUN_TRAINING=True to execute. (Note: full run ~6–10h on T4.)')

### 2.1 Metrics — *to be filled in*
| Run | Stanford val acc | VMMRdb val acc | Anchor make acc | Notes |
|---|---|---|---|---|
| Baseline (Stanford only) | 0.77 | n/a | n/a | release v0.2.0 base |
| VMMRdb continue | TBD | TBD | TBD | release v0.2.0 head-swap |


## 3. Damage segmentation training (CarDD, nc=6)

**Goal:** find the damaged regions and what *type* of damage they are.
This is the workhorse for Variants C and D.

### Data
- **CarDD** — 4,000 images, 6 damage types: `dent · scratch · crack ·
  glass-shatter · lamp-broken · tire-flat`. Permissive license.
- Converted to YOLO segmentation format via `ccdp train build-yolo-dataset`
  (polygon masks → `[class x1 y1 x2 y2 …]` lines).

### Model
YOLOv8-small segmentation head (`yolov8s-seg.pt`). Small > nano for the modest
extra accuracy; we keep imgsz at 640 since damage features are small.

### Result on the released weights
- Box mAP50 = **0.712**, Mask mAP50 = **0.711**.

The training command below produced those numbers. Guarded; smoke values default.

In [ ]:
# === Section 3: Damage YOLOv8-seg on CarDD ===
RUN_TRAINING = False

# Production values (release v0.2.0):
#   model='yolov8s-seg.pt', epochs=80, batch=16, imgsz=640, patience=20
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)

if RUN_TRAINING:
    cmd = (
        f"ccdp train detector --model yolov8s-seg.pt --tag damage_seg "
        f"--epochs {SMOKE['epochs']} --batch {SMOKE['batch']} "
        f"--imgsz {SMOKE['imgsz']} --patience {SMOKE['patience']}"
    )
    print('$', cmd); os.system(cmd)
else:
    print('Skipped — production took ~2h on a T4 with the full settings.')

## 3b. Path A extension — CarDD + HITL (nc=6, warm-start)

> **🚧 Optional. Remove this whole section if not run.**

**Premise.** CarDD is small (~4k images). HITL is another CC0 damage dataset
(~1,812 images, 8 damage classes). Path A keeps the existing `nc=6` head and
warm-starts from the released `damage_seg.pt`, adding HITL examples for the
6 overlapping classes only. No head re-init, no big LR warmup — just more
data for the same labels.

(Path B — extending to `nc=8` to also predict HITL's two new classes — is
not pursued in this submission; the head re-init costs ~2-3× more epochs
and isn't justified given the catalog only prices the 6 CarDD classes.)

### Status
This was kicked off via `notebooks/train_damage_seg_hitl_pathA.ipynb`.
- ☐ Not yet trained → **delete this section before submission.**
- ☑ Trained → fill in the metrics table below.

### Metrics — *to be filled in if run*
| Model | Box mAP50 | Mask mAP50 | Notes |
|---|---|---|---|
| Damage seg (CarDD only, baseline) | 0.712 | 0.711 | release v0.2.0 |
| Damage seg (CarDD + HITL Path A) | TBD | TBD | warm-start, nc=6 |


In [ ]:
# === Section 3b: Path A warm-start ===
RUN_TRAINING = False

# Production values (Path A target):
#   epochs=40, batch=16, imgsz=640, patience=15
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)

if RUN_TRAINING:
    # Assumes the combined CarDD+HITL data.yaml has been built per the
    # path-A notebook (notebooks/train_damage_seg_hitl_pathA.ipynb).
    from ultralytics import YOLO
    import pathlib
    data_yaml = pathlib.Path('data/processed/cardd_hitl_pathA/data.yaml')
    assert data_yaml.exists(), 'Run sections 5–6 of train_damage_seg_hitl_pathA.ipynb first.'
    model = YOLO('checkpoints/production/damage_seg.pt')
    model.train(data=str(data_yaml.resolve()), **SMOKE,
                project='checkpoints/segmenter', name='pathA_submission',
                exist_ok=True, plots=False, verbose=False)
else:
    print('Skipped. If Path A is not run, delete this whole section before submission.')

## 4. Parts segmentation training (carparts, nc=15)

**Goal:** know *which panel* each damage mask lives on.
Without this, we can't say "left front bumper severe scratch" — we can only
say "scratch somewhere on the car," which isn't actionable for costing.

### Data
- **carparts** — CC0 segmentation dataset, ~7k images, 15 canonical parts:
  `front-bumper · rear-bumper · front-door · rear-door · fender · hood ·
  trunk · roof · windshield · rear-window · side-mirror · headlight ·
  taillight · wheel · grille`.

### Model
Same backbone (`yolov8s-seg.pt`), independent training run.

### Result on the released weights
- Box mAP50 = **0.704**, Mask mAP50 = **0.714**.


In [ ]:
# === Section 4: Parts YOLOv8-seg on carparts ===
RUN_TRAINING = False

# Production values (release v0.2.0):
#   model='yolov8s-seg.pt', epochs=80, batch=16, imgsz=640, patience=20, data='carparts.yaml'
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)

if RUN_TRAINING:
    from ultralytics import YOLO
    # Substitute the path to your carparts data.yaml when training fresh.
    data_yaml = 'data/processed/carparts/data.yaml'
    model = YOLO('yolov8s-seg.pt')
    model.train(data=data_yaml, **SMOKE,
                project='checkpoints/segmenter', name='parts_seg',
                exist_ok=True, plots=False, verbose=False)
else:
    print('Skipped — production took ~2h on a T4.')

## 5. Variant A — ResNet-50 multilabel damage classifier

> *The naive first pass: "what kind of damage is in this image?"*

### What we did
Fine-tuned ResNet-50 on CarDD as a multilabel classifier — six independent
binary heads (`dent`, `scratch`, `crack`, …). For each detected damage type,
look up a flat catalog price and sum.

### What's missing
- **No location.** "Scratch present" doesn't tell you which panel.
- **No size.** Light scratch on a fender ≈ ₹1.5k. Deep scratch across the
  hood ≈ ₹18k. The classifier can't distinguish them.
- **No multi-instance.** Two dents on the same panel cost more than one.

### Inference example

In [ ]:
from ccdp.infer.variant_a import classify
# variant_a.classify(path) -> {'damage_types': [...], 'flat_estimate_usd': float}
# Smoke run on a bundled sample image (skip if not present):
import pathlib
sample = pathlib.Path('docs/assets/sample_damage.jpg')
if sample.exists():
    out = classify(str(sample))
    print(out)
else:
    print('(no sample image bundled — Variant A inference shown only schematically)')

## 6. Variant B — YOLOv8 box detector + XGBoost on bbox features

> *"Where is the damage?" — add location and rough size.*

### What we added
- **YOLOv8 box detector** on CarDD: bounding boxes + damage-type labels.
- **XGBoost regressor** trained on per-image features derived from the
  detector's output: damage-type counts, total bbox area %, max bbox area,
  damage spread (centre of mass), etc. → predicts cost in USD.

### Result
- **Real-detector R² = 0.736** on CarDD's test split, RMSE ≈ \$220.

### What's still missing
- **Bbox area is a noisy proxy for damage size.** A scratch and a crack with
  the same bounding box have wildly different cost.
- **No part attribution.** We still don't know which panel.
- **The regressor is a black box.** Hard to explain "why ₹40k?" to an
  insurer or customer.

In [ ]:
from ccdp.infer.variant_b import estimate_cost
sample = pathlib.Path('docs/assets/sample_damage.jpg')
if sample.exists():
    print(estimate_cost(str(sample)))
else:
    print('(schematic: Variant B returns {damage_boxes, predicted_usd})')

## 7. Variant C — YOLOv8-seg + XGBoost on mask features

> *"How big is the damage really?" — add shape/area.*

### What we added
Switched the detector to **YOLOv8-seg** so we get pixel masks instead of
boxes. Re-trained XGBoost on per-image mask-area features.

### The lesson — *this is the section that taught us the deepest thing*
Two XGBoost models, same architecture, different feature source:

| Features | R² on CarDD test |
|---|---|
| **Ground-truth polygons** | **0.939** |
| **Real model predictions** | **0.709** |

The ~0.23 R² gap is the noise of the segmenter's mask quality. Even though
masks **are** more informative than bboxes (the GT score proves it), the real
masks aren't clean enough to beat Variant B's R²=0.736 — they're roughly tied.

**Insight: "how you *use* the information matters as much as having it."**
Variant C had everything Variant D would have (masks + parts). But it fed
those into an opaque regressor that learnt to predict a single scalar cost.
The regressor's error swallowed all the gain. So we changed *how* we used
the masks → Variant D.

## 8. Variant D — Production pipeline (catalog line items)

> *Use the masks directly. Skip the regressor entirely.*

### The pipeline
For each pixel where damage and part masks both fire, we know:
- the **canonical part** (e.g. `front-bumper`),
- the **damage type** (e.g. `crack`),
- the **severity** (`minor / moderate / severe`, bucketed by mask-area ratio):
  - severe ≥ 0.06 of car-area
  - moderate ≥ 0.015
  - else minor.

Then we look up the 3-tier catalog: `(part × severity)` → cost band.

### Why this beats Variant C
- **No regressor.** No black-box error.
- **Per-line-item output.** Each row is a damaged part and a price — auditable
  by a human adjuster. *"Front bumper, moderate crack, ₹14,500."*
- **The masks don't have to be perfect.** They just have to fire in the right
  region. Even a 0.71 mAP50 mask is plenty for "the damage overlaps with the
  front bumper polygon → flag this part."

### What "got better A → B → C → D"
| Variant | What it added | What it still missed |
|---|---|---|
| A — classifier  | damage **type** | location, size, instance count |
| B — det + xgb   | location (bbox), rough size | shape, part, opaque cost |
| C — seg + xgb   | **shape** (mask) | same regressor limit; opaque |
| D — seg ∩ parts | **part attribution, severity, auditable line items** | (this is what we shipped) |


In [ ]:
from ccdp.infer.variant_d import estimate
sample = pathlib.Path('docs/assets/sample_damage.jpg')
if sample.exists():
    report = estimate(str(sample))
    for line in report['line_items']:
        print(f"  {line['part']:18s} {line['damage_type']:14s} "
              f"{line['severity']:8s} ${line['cost_usd']:.0f}")
    print(f"Total: ${report['total_usd']:.0f}")
else:
    print('(schematic: Variant D returns per-part line items + total)')

## 9. Multi-car extension

A single photo might show two cars in a collision. The pipeline above runs
**per car**. We use a Mask R-CNN gate (`detect_all`) to crop each car
instance, then apply identification + Variant D to each crop independently.

The final report aggregates **per-vehicle** line items, so the adjuster can
see "Vehicle 1 (Honda City): ₹38k. Vehicle 2 (Maruti Swift): ₹22k."


In [ ]:
from ccdp.infer.multi_car import estimate_multi
sample = pathlib.Path('docs/assets/sample_collision.jpg')
if sample.exists():
    print(estimate_multi(str(sample)))
else:
    print('(schematic: multi-car returns a list of per-vehicle reports)')

## 10. Live inference demo

Upload a damage photo. The pipeline runs all stages and returns the line-itemed
cost report in both USD and INR.

> Run sections 1.1–1.3 first to fetch the released weights.

In [ ]:
# Upload widget — works on Colab; on plain Jupyter, edit the path below.
import os, sys, pathlib

IMG_PATH = None  # set this to a local path to skip the upload widget

if IMG_PATH is None and 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    IMG_PATH = next(iter(uploaded))
elif IMG_PATH is None:
    IMG_PATH = 'docs/assets/sample_damage.jpg'   # fall back to bundled sample

print(f'inference image: {IMG_PATH}')
assert pathlib.Path(IMG_PATH).exists(), f'no file at {IMG_PATH}'

In [ ]:
# End-to-end Variant D + multi-car run.
from ccdp.infer.multi_car import estimate_multi
import json

report = estimate_multi(IMG_PATH)
print(json.dumps(report, indent=2, default=str))

In [ ]:
# Pretty-print the line items + INR conversion.
USD_TO_INR = 83.0
for i, veh in enumerate(report.get('vehicles', [report]), start=1):
    print(f'\n=== Vehicle {i}: {veh.get("identification", {}).get("class", "unknown")} ===')
    total_usd = 0.0
    for line in veh.get('line_items', []):
        usd = float(line['cost_usd']); total_usd += usd
        inr = usd * USD_TO_INR
        print(f"  {line['part']:18s} {line['damage_type']:14s} "
              f"{line['severity']:8s}  ${usd:>7.0f}  ₹{inr:>9,.0f}")
    print(f"  {'TOTAL':36s}  ${total_usd:>7.0f}  ₹{total_usd*USD_TO_INR:>9,.0f}")

In [ ]:
# Visualize: original image + damage masks + parts masks overlay
import matplotlib.pyplot as plt
from PIL import Image
from ccdp.infer.seg_inference import visualize_overlay   # bundled in repo

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(Image.open(IMG_PATH)); ax[0].set_title('input'); ax[0].axis('off')
overlay = visualize_overlay(IMG_PATH)   # returns PIL Image with masks burned in
ax[1].imshow(overlay); ax[1].set_title('damage ∩ parts overlay'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 11. Reproducibility + final metrics

### Released v0.2.0 model card

| Model | Variant role | Backbone | Train data | Val metric | Notes |
|---|---|---|---|---|---|
| `identifier.pt` | identification | ResNet-50 | Stanford-Cars + VMMRdb continue | Stanford 0.77 / VMMRdb TBD | self-describing checkpoint (class_names embedded) |
| `damage_cls.pt` | Variant A | ResNet-50 | CarDD | (sec 5) | multilabel head |
| `damage_det.pt` | Variant B | YOLOv8s | CarDD | mAP50 ≈ 0.70 | box detector |
| `damage_seg.pt` | Variants C, D | YOLOv8s-seg | CarDD | box mAP50 0.712 / mask 0.711 | masks |
| `parts_seg.pt`  | Variant D    | YOLOv8s-seg | carparts | box mAP50 0.704 / mask 0.714 | 15-class |

### Variant comparison (CarDD test split)

| Variant | Approach | Metric | Score |
|---|---|---|---|
| A | ResNet-50 multilabel + flat catalog | acc (per-type) | ~0.78 |
| B | YOLOv8 det + XGBoost on bbox feats | R² (cost) | 0.736 |
| C — GT polygons | YOLOv8-seg + XGBoost on mask feats | R² (cost) | **0.939** ¹ |
| C — real masks  | same, real model preds | R² (cost) | 0.709 |
| D — production | seg ∩ parts → catalog line items | line-item match | (auditable, no R² applies) |

¹ The 0.939 vs 0.709 gap is the *core lesson*: mask information beats bbox
information **when** the masks are clean. With noisy real masks, the regressor's
error eats the gain. Variant D side-steps that by skipping the regressor.

### Datasets used (all permissive)
- **Stanford-Cars** — 196 classes, ImageNet license-compatible
- **VMMRdb** — Kaggle CC0 mirror (`prabashwara/vmmrdb-dataset`), ~9,170 classes
- **CarDD** — damage seg, permissive
- **carparts** — 15-class parts seg, CC0
- **HITL** (optional, Path A) — 1,812 images, 8 damage classes, CC0

### How to reproduce end-to-end (production values)
```bash
# 1. Identifier two-stage (Stanford → VMMRdb continue)
ccdp train identifier-continue --dataset vmmrdb --top-n 0 \
    --batch-size 64 --num-workers 4 --epochs-stage1 2 --epochs-stage2 10

# 2. Damage seg (CarDD)
ccdp train detector --model yolov8s-seg.pt --tag damage_seg \
    --epochs 80 --batch 16 --imgsz 640 --patience 20

# 3. Parts seg (carparts) — see section 4
# (data prep documented in src/ccdp/data/carparts.py)

# 4. (Optional) Path A — see notebooks/train_damage_seg_hitl_pathA.ipynb
```

### Repo + release
- Code: <https://github.com/theDocWho/car-crash-fix-amount-predictor>
- Weights: v0.2.0 release of the same repo
- HuggingFace Space (live demo): see repo README
